In [ ]:
% load_ext autoreload
% autoreload 2

import os
import torchio as tio

# pra usar cpu, descomentar linha abaixo
#os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
import math
import nibabel as nib

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, CSVLogger, EarlyStopping
from tensorflow.keras.layers import Input, Conv3D, MaxPooling3D, Flatten, Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2
from tensorflow.keras import layers, models, Input, Model

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import gc
import seaborn as sns

from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from PIL import Image
import tempfile
import random

#import wandb

import utils.processamento_dados as proc_dados
import utils.metricas_e_visualização as met_vil

In [ ]:
from tensorflow.keras import mixed_precision

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)  # Evita uso excessivo de memória
        print("GPU habilitada com sucesso!")
        print("Memory Growth habilitado para a GPU")
    except RuntimeError as e:
        print(e)

mixed_precision.set_global_policy("mixed_float16")

tf.get_logger().setLevel('ERROR')

In [ ]:
# FUNÇÕES

def create_model_3d(input_shape, n_classes):
    inputs = Input(shape=input_shape)  # (D, H, W, C)

    # Camada 1
    x = layers.Conv3D(4, (3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.3)(x)
    x = layers.AveragePooling3D(pool_size=(3, 3, 3), padding='same')(x)
    x = layers.Dropout(0.3)(x)

    # Camada 2
    x = layers.Conv3D(8, (3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.3)(x)
    x = layers.AveragePooling3D(pool_size=(3, 3, 3), padding='same')(x)
    x = layers.Dropout(0.3)(x)

    # Camada 3
    x = layers.Conv3D(16, (3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.3)(x)
    x = layers.AveragePooling3D(pool_size=(3, 3, 3), padding='same')(x)
    x = layers.Dropout(0.3)(x)
    
    # Flatten e densas
    x = layers.Flatten()(x)

    x = layers.Dense(16, kernel_regularizer=l2(0.01))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.LeakyReLU(negative_slope=0.3)(x)

    outputs = layers.Dense(n_classes, activation='softmax')(x)

    model = models.Model(inputs=inputs, outputs=outputs)

    return model

In [ ]:
# Definindo caminhos
dir_base = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn"

# train_dir = f'{dir_base}/ADNI/ADNI_NORMALIZED/train'
# val_dir = f'{dir_base}/ADNI/ADNI_NORMALIZED/validation'
# test_dir = f'{dir_base}/ADNI/ADNI_NORMALIZED/test'

fine_tunning_train_dir = f'{dir_base}/ADNI/ADNI_3_4_NORMALIZED/train'
fine_tunning_val_dir = f'{dir_base}/ADNI/ADNI_3_4_NORMALIZED/validation'
fine_tunning_test_dir = f'{dir_base}/ADNI/ADNI_3_4_NORMALIZED/test'

# oasis_train = f'{dir_base}/OASIS/OASIS_1_FSL_NORMALIZED/train'
# oasis_validation = f'{dir_base}/OASIS/OASIS_1_FSL_NORMALIZED/validation'

results_dir_base = f'{dir_base}/ADNI/ADNI_NORMALIZED/results/ruidos_pre_treino_fixo'
os.makedirs(results_dir_base, exist_ok=True)

transformations = ['noise']

# Nome das classes
adni_class_names = ['cn', 'mci', 'ad']
oasis_class_names = ['0.0', '0.5', '1.0']

n_adni_classes = len(adni_class_names)
n_oasis_classes = len(oasis_class_names)

fine_tunning_class_names = adni_class_names
n_fine_tunning_class_names = n_adni_classes

pretrained_model_path = f"{dir_base}/ADNI/ADNI_NORMALIZED/modelo_pretrained.keras"

# CARREGA O MODELO DA FASE 1
if not os.path.exists(pretrained_model_path):
    raise FileNotFoundError(f"Modelo pré-treinado não encontrado em {pretrained_model_path}. Rode a Fase 1 primeiro.")

print(f"Carregando pesos de: {pretrained_model_path}")
base_model = load_model(pretrained_model_path)

# Transfer learning

In [ ]:
train_images_fine_tunning, train_labels_fine_tunning, train_paths_fine_tunning, class_labels_fine_tunning = proc_dados.load_nifti_data_balanced_full_augment(
    fine_tunning_train_dir, 
    adni_class_names, 
    augment=True
)

print(f"N treino: {len(train_images_fine_tunning)}")

val_images_fine_tunning, val_labels_fine_tunning, val_paths_fine_tunning, _ = proc_dados.load_nifti_data_balanced_full_augment(
    fine_tunning_val_dir, 
    adni_class_names,
    augment_originals=False
)

print(f"N validation: {len(val_paths_fine_tunning)}")

In [ ]:
n = len(os.listdir(results_dir_base))

if (n > 0):
    if (len(os.listdir(os.path.join(results_dir_base, f'test_{n}'))) < 3): 
        for item in os.listdir(os.path.join(results_dir_base, f"test_{n}")):
            os.remove(os.path.join(results_dir_base,  f"test_{n}", item))
        os.removedirs(os.path.join(results_dir_base, f'test_{n}'))
        n -= 1

folder_name = f"test_{str(n+1)}"
results_dir = os.path.join(results_dir_base, folder_name)
os.makedirs(results_dir, exist_ok=True)
print(f"pasta {folder_name} criada")

In [ ]:
met_vil.create_pdf_no_predictions(train_images_fine_tunning, train_labels_fine_tunning, os.path.join(results_dir, 'augment_training_data.pdf'), adni_class_names)

### Predição Validação

In [ ]:
# Realizar predições para dados do conjunto validação
val_pred_labels_post_train, val_true_labels_post_train, val_pred_post_train = met_vil.get_predictions(val_images_fine_tunning, val_labels_fine_tunning, 64, base_model)

os.makedirs(f"{results_dir}/pretrained", exist_ok=True)
# Obter métricas da valiadação e salvá-las em um arquivo
met_vil.get_classification_report(val_true_labels_post_train, val_pred_labels_post_train, f"{results_dir}/pretrained", 'validation_adni_pre_train')

# Obter matriz de confusão
met_vil.plot_confusion_matrix(val_true_labels_post_train, val_pred_labels_post_train, f"{results_dir}/pretrained", 'validation_adni_pre_train', oasis_class_names)

# Criar pdf com predições
val_pdf_path_post_train = os.path.join(f"{results_dir}/pretrained", "validation_adni_predictions_pre_train.pdf")
met_vil.create_pdf(val_paths_fine_tunning, val_images_fine_tunning, val_true_labels_post_train, val_pred_labels_post_train, val_pred_post_train, val_pdf_path_post_train, oasis_class_names)

### Predição Teste

In [ ]:
test_images_fine_tunning, test_labels_fine_tunning, test_paths_fine_tunning, _ = proc_dados.load_nifti_data_balanced_preallocated(fine_tunning_test_dir, adni_class_names)

# Realizar predições para dados do conjunto validação
test_pred_labels_fine_tunning, test_true_labels_fine_tunning, test_pred = met_vil.get_predictions(test_images_fine_tunning, test_labels_fine_tunning, 64, base_model)

# Obter métricas da valiadação e salvá-las em um arquivo
met_vil.get_classification_report(test_true_labels_fine_tunning, test_pred_labels_fine_tunning, f"{results_dir}/pretrained", 'test_adni_pre_trained')

# Obter matriz de confusão
met_vil.plot_confusion_matrix(test_true_labels_fine_tunning, test_pred_labels_fine_tunning, f"{results_dir}/pretrained", 'test_adni_pre_trained', adni_class_names)

# Criar pdf com predições
test_pdf_path_fine_tunning = os.path.join(f"{results_dir}/pretrained", "test_adni_predictions_pre_trained.pdf")
met_vil.create_pdf(test_paths_fine_tunning, test_images_fine_tunning, test_true_labels_fine_tunning, test_pred_labels_fine_tunning, test_pred, test_pdf_path_fine_tunning, adni_class_names)

### Retreinamento

In [ ]:
fase_atual = 2
print("--- INICIANDO FASE 2: FINE-TUNING ---")

# --- CONGELAMENTO E ADAPTAÇÃO ---
# 1. Remover a camada de classificação antiga e adicionar uma nova (ADnetEx)
# Pega a saída da camada anterior à última
x = base_model.layers[-2].output

# Adiciona nova camada de classificação inicializada aleatoriamente
predictions = layers.Dense(n_fine_tune_class_names, activation='softmax', name='new_output')(x)

model = models.Model(inputs=base_model.input, outputs=predictions)

# 2. Congelar camadas iniciais (Freezing)
# Vamos congelar os dois primeiros blocos convolucionais
for layer in model.layers[:10]:
    layer.trainable = False
    print(f"Camada congelada: {layer.name}")

# 3. Recompilar
# Usar mesmo learning_rate do pre_treino
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00005), 
                loss='categorical_crossentropy', 
                metrics=['categorical_accuracy'])

model.summary()

In [ ]:
batch_size = 32

steps_per_epoch = math.ceil(len(train_images_fine_tunning) / batch_size)
validation_steps = math.ceil(len(val_images_fine_tunning) / batch_size)

train_generator_fine_tunning = proc_dados.nifti_data_generator_3d(train_images_fine_tunning, train_labels_fine_tunning, batch_size)
val_generator_fine_tunning = proc_dados.nifti_data_generator_3d(val_images_fine_tunning, val_labels_fine_tunning, batch_size)

# Callbacks
csv_filename = 'log_treino_fase2.csv'
checkpoint_path = os.path.join(results_dir, "final_model.keras")

callbacks_list = [
    EarlyStopping(monitor='val_loss', patience=25, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, verbose=1),
    CSVLogger(os.path.join(results_dir, csv_filename)),
    ModelCheckpoint(filepath=checkpoint_path, monitor='val_categorical_accuracy', save_best_only=True, mode='max')
]

In [ ]:
epochs = 100 # Fine-tuning geralmente requer menos épocas

print(f"Iniciando treinamento FASE {fase_atual} por {epochs} épocas...")

history_fine_tunning = model.fit(
    train_generator_fine_tunning,
    epochs=epochs,
    verbose=1,
    validation_data=val_generator_fine_tunning,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=callbacks_list
)

In [ ]:
# Plotando o histórico de treinamento após o treinamento
os.makedirs(f"{results_dir}/final", exist_ok=True)
met_vil.plot_training_history(history_fine_tunning, f"{results_dir}/final", "fine_tunning_oasis_training_history.png")

### PREDIÇÃO VALIDAÇÃO

In [ ]:
# Realizar predições para dados do conjunto validação
val_pred_labels_post_train, val_true_labels_post_train, val_pred_post_train = met_vil.get_predictions(val_images_fine_tunning, val_labels_fine_tunning, batch_size, model)

os.makedirs(f"{results_dir}/final", exist_ok=True)
# Obter métricas da valiadação e salvá-las em um arquivo
met_vil.get_classification_report(val_true_labels_post_train, val_pred_labels_post_train, f"{results_dir}/final", 'validation_adni_post_train')

# Obter matriz de confusão
met_vil.plot_confusion_matrix(val_true_labels_post_train, val_pred_labels_post_train, f"{results_dir}/final", 'validation_adni_post_train', adni_class_names)

# Criar pdf com predições
val_pdf_path_post_train = os.path.join(f"{results_dir}/final", "validation_adni_predictions_post_train.pdf")
met_vil.create_pdf(val_paths_fine_tunning, val_images_fine_tunning, val_true_labels_post_train, val_pred_labels_post_train, val_pred_post_train, val_pdf_path_post_train, fine_tunning_class_names)

### PREDIÇÃO TESTE

In [ ]:
test_images_fine_tunning, test_labels_fine_tunning, test_paths_fine_tunning, _ = proc_dados.load_nifti_data_balanced_preallocated(fine_tunning_test_dir, adni_class_names)

# Realizar predições para dados do conjunto validação
test_pred_labels_fine_tunning, test_true_labels_fine_tunning, test_pred = met_vil.get_predictions(test_images_fine_tunning, test_labels_fine_tunning, batch_size, model)

# Obter métricas da valiadação e salvá-las em um arquivo
met_vil.get_classification_report(test_true_labels_fine_tunning, test_pred_labels_fine_tunning, f"{results_dir}/final", 'test_adni_fine_tunning')

# Obter matriz de confusão
met_vil.plot_confusion_matrix(test_true_labels_fine_tunning, test_pred_labels_fine_tunning, f"{results_dir}/final", 'test_adni_fine_tunning', adni_class_names)

# Criar pdf com predições
test_pdf_path_fine_tunning = os.path.join(f"{results_dir}/final", "test_adni_predictions_fine_tunning.pdf")
met_vil.create_pdf(test_paths_fine_tunning, test_images_fine_tunning, test_true_labels_fine_tunning, test_pred_labels_fine_tunning, test_pred, test_pdf_path_fine_tunning, fine_tunning_class_names)